<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# 6. Chatbot with Persistent Memory (LangGraph)
**Industry:** Banking

Build a chatbot using LangGraph that remembers conversation history across multiple turns and sessions using SQLite checkpointing.

In [1]:
!pip install langgraph langchain langchain-openai python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
import os
import dotenv
dotenv.load_dotenv(r"D:/Internship/Teach-ai/Backend/.env")
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = AzureChatOpenAI(azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"), api_key=os.environ.get("AZURE_OPENAI_API_KEY"), azure_deployment="gpt-4o", api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"))

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
memory = SqliteSaver(conn)

graph = graph_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "customer_123"}}

# Session 1
user_input = "Hi, my loan application ID is 987654."
events = graph.stream({"messages": [("user", user_input)]}, config)
for event in events:
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

print("\n--- Restarting Script (Session 2) ---\n")

# Session 2
user_input = "Can you tell me what my loan application ID was?"
events = graph.stream({"messages": [("user", user_input)]}, config)
for event in events:
    for value in event.values():
        print("Assistant:", value["messages"][-1].content)

ModuleNotFoundError: No module named 'langgraph.checkpoint.sqlite'